# 从零实现 Vision Transformer：Patch、手写多头注意力与 Encoder

这份 notebook 不使用 `nn.MultiheadAttention`、`nn.Transformer`、`timm` 或 `torchvision.models`。我们从 `PatchEmbedding` 开始，手写 scaled dot-product 多头自注意力、MLP、pre-norm `EncoderBlock` 和完整 `VisionTransformer.forward`，并验证注意力 shape/softmax/梯度、位置长度、微型过拟合、validation 评估、数值注意力摘要与位置插值边界。

所有数据离线合成、全程 CPU。这里复现的是核心计算图，不是 ImageNet 训练配方或论文精度复现。


## 1. 计算图与验收面

```text
image [N,C,H,W]
  -> non-overlapping patches
  -> linear projection -> [N,P,D]
  -> prepend learnable [CLS] -> [N,P+1,D]
  -> add position embedding
  -> L × {LayerNorm -> manual MHSA -> residual -> LayerNorm -> MLP -> residual}
  -> normalized CLS -> classification logits [N,K]
```

必须同时验证：patch 整除、token 数、`D % heads == 0`、attention 的 `[N,H,T,T]`、softmax 行和、位置向量长度、非零有限梯度和分类头 logits。只打印 `model` 不能证明这些契约成立。


In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

from hashlib import sha256  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 240728  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
np.random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.use_deterministic_algorithms(True)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})  # 执行当前语句以推进本节示例。


## 2. Patch embedding：卷积只是共享线性投影的高效写法

若图像为 $H\times W$，patch 为 $P_h\times P_w$ 且两维整除，token 数是 $N_p=(H/P_h)(W/P_w)$。每个 patch 展平后是 $C P_h P_w$ 维，再映射到 embedding 维 $D$。

`Conv2d(C,D,kernel_size=P,stride=P)` 与“切无重叠 patch 后对每块使用同一个线性层”等价。这里没有借用现成 ViT，只用卷积完成共享投影。固定 learned position embedding 意味着运行时尺寸默认必须和训练配置一致。


In [ ]:
class PatchEmbedding(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, image_size=(16, 16), patch_size=(4, 4), in_channels=1, embed_dim=24):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.image_size = tuple(image_size)  # 计算并保存当前步骤的中间状态。
        self.patch_size = tuple(patch_size)  # 计算并保存当前步骤的中间状态。
        if any(size % patch != 0 for size, patch in zip(self.image_size, self.patch_size)):  # 按当前条件选择后续控制路径。
            raise ValueError("image_size 的高宽必须分别被 patch_size 整除")  # 遇到非法合同立即显式失败。
        self.grid_size = tuple(size // patch for size, patch in zip(self.image_size, self.patch_size))  # 计算并保存当前步骤的中间状态。
        self.num_patches = self.grid_size[0] * self.grid_size[1]  # 计算并保存当前步骤的中间状态。
        self.projection = nn.Conv2d(in_channels, embed_dim,  # 计算并保存当前步骤的中间状态。
                                    kernel_size=self.patch_size, stride=self.patch_size)  # 计算并保存当前步骤的中间状态。

    def forward(self, images):  # 定义本节可复用的核心函数。
        if images.ndim != 4:  # 按当前条件选择后续控制路径。
            raise ValueError("PatchEmbedding 期望 NCHW")  # 遇到非法合同立即显式失败。
        if tuple(images.shape[-2:]) != self.image_size:  # 按当前条件选择后续控制路径。
            raise ValueError(f"固定位置编码要求输入 {self.image_size}，收到 {tuple(images.shape[-2:])}")  # 遇到非法合同立即显式失败。
        features = self.projection(images)  # 计算并保存当前步骤的中间状态。
        return features.flatten(2).transpose(1, 2)  # 返回当前分支计算出的结果。

patcher = PatchEmbedding()  # 计算并保存当前步骤的中间状态。
patch_tokens = patcher(torch.zeros(2, 1, 16, 16))  # 计算并保存当前步骤的中间状态。
assert patch_tokens.shape == (2, 16, 24)  # 用受控断言验证关键不变量。
assert patcher.grid_size == (4, 4)  # 用受控断言验证关键不变量。
assert patcher.num_patches == 16  # 用受控断言验证关键不变量。


## 3. 手写多头自注意力

对 token 矩阵 $X\in\mathbb{R}^{T\times D}$，线性映射得到 $Q,K,V$。每个 head 的维度 $d_h=D/H$：

$$A=\mathrm{softmax}(QK^\top/\sqrt{d_h}),\qquad Z=AV.$$

$1/\sqrt{d_h}$ 防止维度变大时点积方差过大、softmax 过度饱和。多头不是重复同一矩阵：reshape 为 `[N,heads,T,head_dim]` 后，每个 head 使用 Q/K/V 的不同子空间。attention 行在最后一维归一化为 1。


In [ ]:
class MultiHeadSelfAttention(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, embed_dim, num_heads, dropout=0.0):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if embed_dim % num_heads != 0:  # 按当前条件选择后续控制路径。
            raise ValueError("embed_dim 必须被 num_heads 整除")  # 遇到非法合同立即显式失败。
        self.embed_dim = embed_dim  # 计算并保存当前步骤的中间状态。
        self.num_heads = num_heads  # 计算并保存当前步骤的中间状态。
        self.head_dim = embed_dim // num_heads  # 计算并保存当前步骤的中间状态。
        self.scale = self.head_dim ** -0.5  # 计算并保存当前步骤的中间状态。
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)  # 计算并保存当前步骤的中间状态。
        self.attn_dropout = nn.Dropout(dropout)  # 计算并保存当前步骤的中间状态。
        self.output = nn.Linear(embed_dim, embed_dim)  # 计算并保存当前步骤的中间状态。
        self.output_dropout = nn.Dropout(dropout)  # 计算并保存当前步骤的中间状态。

    def forward(self, tokens, return_attention=False):  # 定义本节可复用的核心函数。
        if tokens.ndim != 3 or tokens.shape[-1] != self.embed_dim:  # 按当前条件选择后续控制路径。
            raise ValueError("attention 期望 [N,T,D] 且 D 与配置一致")  # 遇到非法合同立即显式失败。
        batch, token_count, _ = tokens.shape  # 计算并保存当前步骤的中间状态。
        qkv = self.qkv(tokens).reshape(batch, token_count, 3,  # 计算并保存当前步骤的中间状态。
                                       self.num_heads, self.head_dim)  # 执行当前语句以推进本节示例。
        qkv = qkv.permute(2, 0, 3, 1, 4)  # 计算并保存当前步骤的中间状态。
        queries, keys, values = qkv.unbind(0)  # 计算并保存当前步骤的中间状态。
        scores = (queries @ keys.transpose(-2, -1)) * self.scale  # 计算并保存当前步骤的中间状态。
        attention = scores.softmax(dim=-1)  # 计算并保存当前步骤的中间状态。
        context = self.attn_dropout(attention) @ values  # 计算并保存当前步骤的中间状态。
        context = context.transpose(1, 2).contiguous().reshape(batch, token_count, self.embed_dim)  # 计算并保存当前步骤的中间状态。
        output = self.output_dropout(self.output(context))  # 计算并保存当前步骤的中间状态。
        return (output, attention) if return_attention else output  # 返回当前分支计算出的结果。

attention_probe = MultiHeadSelfAttention(embed_dim=24, num_heads=4)  # 计算并保存当前步骤的中间状态。
probe_tokens = torch.randn(2, 17, 24, requires_grad=True)  # 计算并保存当前步骤的中间状态。
probe_output, probe_map = attention_probe(probe_tokens, return_attention=True)  # 计算并保存当前步骤的中间状态。
probe_output.square().mean().backward()  # 执行当前语句以推进本节示例。
assert probe_output.shape == probe_tokens.shape  # 用受控断言验证关键不变量。
assert probe_map.shape == (2, 4, 17, 17)  # 用受控断言验证关键不变量。
assert torch.allclose(probe_map.sum(dim=-1), torch.ones(2, 4, 17), atol=1e-6)  # 用受控断言验证关键不变量。
assert probe_tokens.grad is not None and float(probe_tokens.grad.norm()) > 0  # 用受控断言验证关键不变量。
assert torch.isfinite(probe_tokens.grad).all()  # 用受控断言验证关键不变量。

# 数值 oracle：Q/K/V 与输出投影设成恒等映射，显式计算 1/sqrt(d_h)。
scale_oracle = MultiHeadSelfAttention(embed_dim=2, num_heads=1, dropout=0.0).eval()  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    identity2 = torch.eye(2)  # 计算并保存当前步骤的中间状态。
    scale_oracle.qkv.weight.copy_(torch.cat([identity2, identity2, identity2], dim=0))  # 计算并保存当前步骤的中间状态。
    scale_oracle.qkv.bias.zero_()  # 执行当前语句以推进本节示例。
    scale_oracle.output.weight.copy_(identity2)  # 执行当前语句以推进本节示例。
    scale_oracle.output.bias.zero_()  # 执行当前语句以推进本节示例。
oracle_tokens = torch.tensor([[[1.0, 0.0], [0.0, 1.0]]])  # 计算并保存当前步骤的中间状态。
oracle_output, oracle_attention = scale_oracle(oracle_tokens, return_attention=True)  # 计算并保存当前步骤的中间状态。
expected_scores = (oracle_tokens @ oracle_tokens.transpose(-2, -1)) / math.sqrt(2.0)  # 计算并保存当前步骤的中间状态。
expected_attention = expected_scores.softmax(dim=-1).unsqueeze(1)  # 计算并保存当前步骤的中间状态。
expected_output = expected_attention.squeeze(1) @ oracle_tokens  # 计算并保存当前步骤的中间状态。
unscaled_attention = (oracle_tokens @ oracle_tokens.transpose(-2, -1)).softmax(dim=-1).unsqueeze(1)  # 计算并保存当前步骤的中间状态。
assert math.isclose(scale_oracle.scale, 1 / math.sqrt(2.0), rel_tol=1e-12)  # 用受控断言验证关键不变量。
assert torch.allclose(oracle_attention, expected_attention, atol=1e-7)  # 用受控断言验证关键不变量。
assert torch.allclose(oracle_output, expected_output, atol=1e-7)  # 用受控断言验证关键不变量。
assert not torch.allclose(expected_attention, unscaled_attention, atol=1e-4)  # 用受控断言验证关键不变量。


## 4. MLP 与 pre-norm EncoderBlock

Transformer block 包含两条残差：attention 子层和逐 token MLP 子层。pre-norm 写法是

$$x\leftarrow x+\mathrm{MHSA}(\mathrm{LN}(x)),\quad
x\leftarrow x+\mathrm{MLP}(\mathrm{LN}(x)).$$

MLP 对每个 token 独立共享参数，通常先升维再降回 $D$；它不在 token 之间混合信息，token 交互发生在 attention。残差两侧维度保持 `[N,T,D]`。


In [ ]:
class MLP(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, embed_dim, hidden_dim, dropout=0.0):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.net = nn.Sequential(  # 计算并保存当前步骤的中间状态。
            nn.Linear(embed_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),  # 执行当前语句以推进本节示例。
            nn.Linear(hidden_dim, embed_dim), nn.Dropout(dropout),  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.net(x)  # 返回当前分支计算出的结果。

class EncoderBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, embed_dim, num_heads, mlp_ratio=2.0, dropout=0.0):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.norm1 = nn.LayerNorm(embed_dim)  # 计算并保存当前步骤的中间状态。
        self.attention = MultiHeadSelfAttention(embed_dim, num_heads, dropout)  # 计算并保存当前步骤的中间状态。
        self.norm2 = nn.LayerNorm(embed_dim)  # 计算并保存当前步骤的中间状态。
        self.mlp = MLP(embed_dim, int(embed_dim * mlp_ratio), dropout)  # 计算并保存当前步骤的中间状态。

    def forward(self, x, return_attention=False):  # 定义本节可复用的核心函数。
        attention_output, attention_map = self.attention(self.norm1(x), return_attention=True)  # 计算并保存当前步骤的中间状态。
        x = x + attention_output  # 计算并保存当前步骤的中间状态。
        x = x + self.mlp(self.norm2(x))  # 计算并保存当前步骤的中间状态。
        return (x, attention_map) if return_attention else x  # 返回当前分支计算出的结果。

block_probe = EncoderBlock(24, 4)  # 计算并保存当前步骤的中间状态。
block_output, block_map = block_probe(torch.zeros(2, 17, 24), return_attention=True)  # 计算并保存当前步骤的中间状态。
assert block_output.shape == (2, 17, 24)  # 用受控断言验证关键不变量。
assert block_map.shape[-2:] == (17, 17)  # 用受控断言验证关键不变量。
assert sum(isinstance(module, nn.GELU) for module in block_probe.modules()) == 1  # 用受控断言验证关键不变量。


## 5. class token、位置向量与完整 VisionTransformer

所有 patch projection 共享权重，因此单靠内容无法知道 patch 位于哪里；learned position embedding 为每个序号加入位置信息。可学习 `[CLS]` token 放在序列首位，通过多层 attention 汇聚 patch 信息，最终分类头读取它。

`pos_embed` 长度必须是 `num_patches + 1`。下面将 token 装配拆成 `add_class_and_position`，让长度错误可以被独立测试，而不是等矩阵广播报出难懂异常。


In [ ]:
class VisionTransformer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, image_size=(16, 16), patch_size=(4, 4), in_channels=1,  # 定义本节可复用的核心函数。
                 num_classes=3, embed_dim=24, depth=2, num_heads=4,  # 计算并保存当前步骤的中间状态。
                 mlp_ratio=2.0, dropout=0.0):  # 计算并保存当前步骤的中间状态。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)  # 计算并保存当前步骤的中间状态。
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))  # 计算并保存当前步骤的中间状态。
        self.pos_embed = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches + 1, embed_dim))  # 计算并保存当前步骤的中间状态。
        self.pos_dropout = nn.Dropout(dropout)  # 计算并保存当前步骤的中间状态。
        self.blocks = nn.ModuleList([  # 计算并保存当前步骤的中间状态。
            EncoderBlock(embed_dim, num_heads, mlp_ratio, dropout) for _ in range(depth)  # 执行当前语句以推进本节示例。
        ])  # 执行当前语句以推进本节示例。
        self.norm = nn.LayerNorm(embed_dim)  # 计算并保存当前步骤的中间状态。
        self.head = nn.Linear(embed_dim, num_classes)  # 计算并保存当前步骤的中间状态。
        nn.init.trunc_normal_(self.cls_token, std=0.02)  # 计算并保存当前步骤的中间状态。
        nn.init.trunc_normal_(self.pos_embed, std=0.02)  # 计算并保存当前步骤的中间状态。
        self.apply(self._init_weights)  # 执行当前语句以推进本节示例。

    @staticmethod  # 为下方定义附加声明式配置。
    def _init_weights(module):  # 定义本节可复用的核心函数。
        if isinstance(module, nn.Linear):  # 按当前条件选择后续控制路径。
            nn.init.trunc_normal_(module.weight, std=0.02)  # 计算并保存当前步骤的中间状态。
            if module.bias is not None:  # 按当前条件选择后续控制路径。
                nn.init.zeros_(module.bias)  # 执行当前语句以推进本节示例。

    def add_class_and_position(self, patch_tokens):  # 定义本节可复用的核心函数。
        batch = patch_tokens.shape[0]  # 计算并保存当前步骤的中间状态。
        cls = self.cls_token.expand(batch, -1, -1)  # 计算并保存当前步骤的中间状态。
        tokens = torch.cat([cls, patch_tokens], dim=1)  # 计算并保存当前步骤的中间状态。
        if tokens.shape[1:] != self.pos_embed.shape[1:]:  # 按当前条件选择后续控制路径。
            raise ValueError(f"token/position shape 不一致: {tokens.shape} vs {self.pos_embed.shape}")  # 遇到非法合同立即显式失败。
        return self.pos_dropout(tokens + self.pos_embed)  # 返回当前分支计算出的结果。

    def forward(self, images, return_attention=False):  # 定义本节可复用的核心函数。
        x = self.add_class_and_position(self.patch_embed(images))  # 计算并保存当前步骤的中间状态。
        attention_maps = []  # 计算并保存当前步骤的中间状态。
        for block in self.blocks:  # 遍历输入元素以累积或检查结果。
            x, attention = block(x, return_attention=True)  # 计算并保存当前步骤的中间状态。
            attention_maps.append(attention)  # 执行当前语句以推进本节示例。
        logits = self.head(self.norm(x)[:, 0])  # 计算并保存当前步骤的中间状态。
        return (logits, attention_maps) if return_attention else logits  # 返回当前分支计算出的结果。

vit = VisionTransformer().to(DEVICE)  # 计算并保存当前步骤的中间状态。
vit_logits, vit_maps = vit(torch.zeros(2, 1, 16, 16), return_attention=True)  # 计算并保存当前步骤的中间状态。
assert vit_logits.shape == (2, 3)  # 用受控断言验证关键不变量。
assert len(vit_maps) == 2  # 用受控断言验证关键不变量。
assert all(attention.shape == (2, 4, 17, 17) for attention in vit_maps)  # 用受控断言验证关键不变量。


## 6. 参数量与 shape 诊断

attention 的 QKV 权重约为 $3D^2$，输出投影约为 $D^2$，MLP 在 ratio 为 $r$ 时约为 $2rD^2$；token 数不改变这些参数量，却使 attention 激活和计算随 $T^2$ 增长。因此高分辨率的主要风险常是显存/延迟，而不是权重文件大小。

下面用 hook 记录 patch projection、每层 block 和分类头 shape。参数量按子系统聚合，便于发现配置是否真的生效。


In [ ]:
parameter_groups24 = {  # 计算并保存当前步骤的中间状态。
    "patch": sum(p.numel() for p in vit.patch_embed.parameters()),  # 执行当前语句以推进本节示例。
    "tokens": vit.cls_token.numel() + vit.pos_embed.numel(),  # 执行当前语句以推进本节示例。
    "encoder": sum(p.numel() for p in vit.blocks.parameters()),  # 执行当前语句以推进本节示例。
    "head": sum(p.numel() for p in vit.head.parameters()),  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
total_parameters24 = sum(p.numel() for p in vit.parameters())  # 计算并保存当前步骤的中间状态。

shape_trace24 = []  # 计算并保存当前步骤的中间状态。
handles24 = [  # 计算并保存当前步骤的中间状态。
    vit.patch_embed.register_forward_hook(  # 执行当前语句以推进本节示例。
        lambda module, inputs, output: shape_trace24.append(("patch", tuple(output.shape)))),  # 执行当前语句以推进本节示例。
    vit.blocks[0].register_forward_hook(  # 执行当前语句以推进本节示例。
        lambda module, inputs, output: shape_trace24.append(("block0", tuple(output[0].shape)))),  # 执行当前语句以推进本节示例。
    vit.head.register_forward_hook(  # 执行当前语句以推进本节示例。
        lambda module, inputs, output: shape_trace24.append(("head", tuple(output.shape)))),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
vit.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    _ = vit(torch.zeros(3, 1, 16, 16))  # 计算并保存当前步骤的中间状态。
for handle in handles24:  # 遍历输入元素以累积或检查结果。
    handle.remove()  # 执行当前语句以推进本节示例。

assert sum(parameter_groups24.values()) + sum(p.numel() for p in vit.norm.parameters()) == total_parameters24  # 用受控断言验证关键不变量。
assert 10_000 < total_parameters24 < 100_000  # 用受控断言验证关键不变量。
assert shape_trace24 == [("patch", (3, 16, 24)), ("block0", (3, 17, 24)), ("head", (3, 3))]  # 用受控断言验证关键不变量。
print({"groups": parameter_groups24, "total": total_parameters24, "trace": shape_trace24})  # 执行当前语句以推进本节示例。


## 7. 失败反例：patch 整除、head 维度和位置长度

三类错误应在入口尽早失败：

1. 图像高宽不能被 patch 整除，会丢边或产生不一致 grid；
2. embedding 维不能被 head 数整除，无法等分 head_dim；
3. token 数与 learned position 长度不同，不能逐位置相加。

生产接口可选择拒绝未知尺寸，或显式 pad/resize/插值；不能静默截断。


In [ ]:
failure_messages24 = []  # 计算并保存当前步骤的中间状态。
for factory in [  # 遍历输入元素以累积或检查结果。
    lambda: PatchEmbedding(image_size=(15, 16), patch_size=(4, 4)),  # 计算并保存当前步骤的中间状态。
    lambda: MultiHeadSelfAttention(embed_dim=25, num_heads=4),  # 计算并保存当前步骤的中间状态。
]:  # 执行当前语句以推进本节示例。
    try:  # 尝试执行可能失败的受控操作。
        factory()  # 执行当前语句以推进本节示例。
    except ValueError as exc:  # 捕获预期异常并验证失败分支。
        failure_messages24.append(str(exc))  # 执行当前语句以推进本节示例。

try:  # 尝试执行可能失败的受控操作。
    vit.add_class_and_position(torch.zeros(1, 15, 24))  # 执行当前语句以推进本节示例。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    failure_messages24.append(str(exc))  # 执行当前语句以推进本节示例。

try:  # 尝试执行可能失败的受控操作。
    vit(torch.zeros(1, 1, 18, 16))  # 执行当前语句以推进本节示例。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    failure_messages24.append(str(exc))  # 执行当前语句以推进本节示例。

assert len(failure_messages24) == 4  # 用受控断言验证关键不变量。
assert "整除" in failure_messages24[0]  # 用受控断言验证关键不变量。
assert "num_heads" in failure_messages24[1]  # 用受控断言验证关键不变量。
assert "position" in failure_messages24[2]  # 用受控断言验证关键不变量。
assert "输入" in failure_messages24[3]  # 用受控断言验证关键不变量。
print(failure_messages24)  # 执行当前语句以推进本节示例。


## 8. 可控 toy 图像分类

三类图像分别在左上、右上、下中区域放置亮方块，并加入背景噪声和小幅位置扰动。内容近似相同、位置不同，因此任务确实需要保留空间位置信息。训练与 validation 使用独立 seed。

真实图像的纹理、尺度、遮挡和类别语义远复杂得多；ViT 通常还依赖大数据、强增强、正则化和合适训练策略。toy 成功不意味着 ViT 在小数据上总优于 CNN。


In [ ]:
def make_position_images(n_per_class, seed):  # 定义本节可复用的核心函数。
    generator = torch.Generator().manual_seed(seed)  # 计算并保存当前步骤的中间状态。
    anchors = [(3, 3), (3, 11), (11, 7)]  # 计算并保存当前步骤的中间状态。
    images, labels = [], []  # 计算并保存当前步骤的中间状态。
    for label, (base_y, base_x) in enumerate(anchors):  # 遍历输入元素以累积或检查结果。
        for _ in range(n_per_class):  # 遍历输入元素以累积或检查结果。
            jitter_y = int(torch.randint(-1, 2, (1,), generator=generator))  # 计算并保存当前步骤的中间状态。
            jitter_x = int(torch.randint(-1, 2, (1,), generator=generator))  # 计算并保存当前步骤的中间状态。
            image = 0.06 * torch.randn((1, 16, 16), generator=generator)  # 计算并保存当前步骤的中间状态。
            y, x = base_y + jitter_y, base_x + jitter_x  # 计算并保存当前步骤的中间状态。
            image[:, y:y+4, x:x+4] += 1.0  # 计算并保存当前步骤的中间状态。
            images.append(image.clamp(0, 1)); labels.append(label)  # 执行当前语句以推进本节示例。
    permutation = torch.randperm(len(labels), generator=generator)  # 计算并保存当前步骤的中间状态。
    return torch.stack(images)[permutation], torch.tensor(labels, dtype=torch.long)[permutation]  # 返回当前分支计算出的结果。

train_x24, train_y24 = make_position_images(10, SEED + 1)  # 计算并保存当前步骤的中间状态。
val_x24, val_y24 = make_position_images(5, SEED + 2)  # 计算并保存当前步骤的中间状态。
assert train_x24.shape == (30, 1, 16, 16)  # 用受控断言验证关键不变量。
assert train_y24.bincount().tolist() == [10, 10, 10]  # 用受控断言验证关键不变量。
assert not torch.equal(train_x24[:15], val_x24)  # 用受控断言验证关键不变量。
assert 0.0 <= float(train_x24.min()) <= float(train_x24.max()) <= 1.0  # 用受控断言验证关键不变量。


## 9. 小样本过拟合与优化器

使用 `CrossEntropyLoss` 时模型返回原始 `[N,K]` logits，不提前 softmax。AdamW 将权重衰减从自适应梯度更新中解耦；真实 ViT 常对 bias、LayerNorm 和 token 参数采用不同 decay 分组，这里为保持教学链路简洁使用单一参数组。

训练反复使用 30 个样本，只验证计算图能学会位置任务。记录初始/最终 loss 和训练准确率，不把 validation 当生产指标。


In [ ]:
torch.manual_seed(SEED + 10)  # 执行当前语句以推进本节示例。
model24 = VisionTransformer(embed_dim=24, depth=2, num_heads=4,  # 计算并保存当前步骤的中间状态。
                            mlp_ratio=2.0, dropout=0.0).to(DEVICE)  # 计算并保存当前步骤的中间状态。
optimizer24 = torch.optim.AdamW(model24.parameters(), lr=0.01, weight_decay=0.01)  # 计算并保存当前步骤的中间状态。
criterion24 = nn.CrossEntropyLoss()  # 计算并保存当前步骤的中间状态。

model24.train()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    initial_loss24 = float(criterion24(model24(train_x24), train_y24))  # 计算并保存当前步骤的中间状态。
loss_curve24 = []  # 计算并保存当前步骤的中间状态。
for step in range(120):  # 遍历输入元素以累积或检查结果。
    optimizer24.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    logits24 = model24(train_x24)  # 计算并保存当前步骤的中间状态。
    loss24 = criterion24(logits24, train_y24)  # 计算并保存当前步骤的中间状态。
    loss24.backward()  # 执行当前语句以推进本节示例。
    torch.nn.utils.clip_grad_norm_(model24.parameters(), max_norm=5.0)  # 计算并保存当前步骤的中间状态。
    optimizer24.step()  # 执行当前语句以推进本节示例。
    loss_curve24.append(float(loss24.detach()))  # 执行当前语句以推进本节示例。

model24.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    final_train_logits24 = model24(train_x24)  # 计算并保存当前步骤的中间状态。
    final_loss24 = float(criterion24(final_train_logits24, train_y24))  # 计算并保存当前步骤的中间状态。
    train_accuracy24 = float((final_train_logits24.argmax(1) == train_y24).float().mean())  # 计算并保存当前步骤的中间状态。

assert final_loss24 < initial_loss24 * 0.20  # 用受控断言验证关键不变量。
assert train_accuracy24 >= 0.95  # 用受控断言验证关键不变量。
assert len(loss_curve24) == 120  # 用受控断言验证关键不变量。
assert all(math.isfinite(value) for value in loss_curve24)  # 用受控断言验证关键不变量。
print({"initial_loss": round(initial_loss24, 4), "final_loss": round(final_loss24, 4),  # 执行当前语句以推进本节示例。
       "train_accuracy": train_accuracy24})  # 执行当前语句以推进本节示例。


## 10. 独立 validation 与分类评估

训练结束后切到 `eval()`，冻结权重，在独立 validation 上报告 accuracy、逐类 recall 和混淆矩阵。toy 数据平衡，所以 accuracy 尚可；真实长尾任务应增加 macro-F1、balanced accuracy、校准、置信拒识和业务代价。

本节没有 test 集，也没有据 validation 反复修改网络直到得到某个数字；输出只用于检测明显实现退化。


In [ ]:
def confusion_matrix_torch(y_true, y_pred, num_classes):  # 定义本节可复用的核心函数。
    matrix = torch.zeros(num_classes, num_classes, dtype=torch.int64)  # 计算并保存当前步骤的中间状态。
    for truth, prediction in zip(y_true.tolist(), y_pred.tolist()):  # 遍历输入元素以累积或检查结果。
        matrix[truth, prediction] += 1  # 计算并保存当前步骤的中间状态。
    return matrix  # 返回当前分支计算出的结果。

model24.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    val_logits24, val_attention24 = model24(val_x24, return_attention=True)  # 计算并保存当前步骤的中间状态。
    val_prediction24 = val_logits24.argmax(1)  # 计算并保存当前步骤的中间状态。
confusion24 = confusion_matrix_torch(val_y24, val_prediction24, 3)  # 计算并保存当前步骤的中间状态。
recall24 = confusion24.diag() / confusion24.sum(dim=1).clamp_min(1)  # 计算并保存当前步骤的中间状态。
validation_accuracy24 = float((val_prediction24 == val_y24).float().mean())  # 计算并保存当前步骤的中间状态。

assert val_logits24.shape == (15, 3)  # 用受控断言验证关键不变量。
assert confusion24.sum().item() == len(val_y24)  # 用受控断言验证关键不变量。
assert recall24.shape == (3,)  # 用受控断言验证关键不变量。
assert torch.isfinite(recall24).all()  # 用受控断言验证关键不变量。
assert len(val_attention24) == 2  # 用受控断言验证关键不变量。
print({"validation_accuracy": validation_accuracy24,  # 执行当前语句以推进本节示例。
       "per_class_recall": recall24.tolist(), "confusion": confusion24.tolist()})  # 执行当前语句以推进本节示例。


## 11. 注意力可视化的数值摘要

对一个样本，取最后一层 `[CLS]` 查询到 16 个 patch 的 attention，先跨 head 平均，再去掉 `[CLS] -> [CLS]` 权重并重新归一化，reshape 成 4×4 网格。这里输出网格、熵和 top patch，等价于可视化前的数值检查。

注意力权重不是因果解释：多层残差、value 投影、MLP 和 head 混合都会影响预测。若做解释评估，应比较遮挡、梯度方法、attention rollout，并检查稳定性与忠实度，不能把一张热图当结论。


In [ ]:
sample24 = val_x24[:1]  # 计算并保存当前步骤的中间状态。
model24.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    sample_logits24, sample_maps24 = model24(sample24, return_attention=True)  # 计算并保存当前步骤的中间状态。
last_attention24 = sample_maps24[-1][0]             # [heads, tokens, tokens]；中文说明：该行遵循既定约束。
cls_to_patches24 = last_attention24[:, 0, 1:].mean(dim=0)  # 计算并保存当前步骤的中间状态。
cls_to_patches24 = cls_to_patches24 / cls_to_patches24.sum()  # 计算并保存当前步骤的中间状态。
attention_grid24 = cls_to_patches24.reshape(model24.patch_embed.grid_size)  # 计算并保存当前步骤的中间状态。
entropy24 = float(-(cls_to_patches24 * cls_to_patches24.clamp_min(1e-12).log()).sum())  # 计算并保存当前步骤的中间状态。
top_patch24 = int(cls_to_patches24.argmax())  # 计算并保存当前步骤的中间状态。
top_row24, top_col24 = divmod(top_patch24, model24.patch_embed.grid_size[1])  # 计算并保存当前步骤的中间状态。

assert attention_grid24.shape == (4, 4)  # 用受控断言验证关键不变量。
assert torch.isclose(attention_grid24.sum(), torch.tensor(1.0), atol=1e-6)  # 用受控断言验证关键不变量。
assert float(attention_grid24.min()) >= 0.0  # 用受控断言验证关键不变量。
assert 0.0 <= entropy24 <= math.log(16) + 1e-6  # 用受控断言验证关键不变量。
assert 0 <= top_row24 < 4 and 0 <= top_col24 < 4  # 用受控断言验证关键不变量。
print({"prediction": int(sample_logits24.argmax(1)), "target": int(val_y24[0]),  # 执行当前语句以推进本节示例。
       "grid": attention_grid24.numpy().round(4).tolist(),  # 执行当前语句以推进本节示例。
       "entropy": round(entropy24, 4), "top_patch": [top_row24, top_col24]})  # 执行当前语句以推进本节示例。


## 12. 分辨率变化与位置向量插值

learned position embedding 绑定训练时的 patch grid。迁移到新分辨率时，常保留 `[CLS]` 位置向量，把 patch 部分 reshape 为二维网格，用 bicubic 插值到新 grid，再展平拼回。

这只是参数适配策略：新高宽仍须被 patch 整除，插值可能改变空间先验，非方形 grid 必须携带原高宽，超出训练尺度还会带来分布偏移。生产需要在目标分辨率重新验证精度、延迟和显存，不能因 shape 能运行就认为语义等价。


In [ ]:
def resize_position_embedding(position, old_grid, new_grid):  # 定义本节可复用的核心函数。
    if position.ndim != 3 or position.shape[0] != 1:  # 按当前条件选择后续控制路径。
        raise ValueError("position 期望 [1,T,D]")  # 遇到非法合同立即显式失败。
    if position.shape[1] != 1 + old_grid[0] * old_grid[1]:  # 按当前条件选择后续控制路径。
        raise ValueError("old_grid 与 position token 数不匹配")  # 遇到非法合同立即显式失败。
    cls_position = position[:, :1]  # 计算并保存当前步骤的中间状态。
    patch_position = position[:, 1:]  # 计算并保存当前步骤的中间状态。
    patch_position = patch_position.reshape(1, old_grid[0], old_grid[1], -1).permute(0, 3, 1, 2)  # 计算并保存当前步骤的中间状态。
    resized = F.interpolate(patch_position, size=new_grid, mode="bicubic", align_corners=False)  # 计算并保存当前步骤的中间状态。
    resized = resized.permute(0, 2, 3, 1).reshape(1, new_grid[0] * new_grid[1], -1)  # 计算并保存当前步骤的中间状态。
    return torch.cat([cls_position, resized], dim=1)  # 返回当前分支计算出的结果。

resized_position24 = resize_position_embedding(model24.pos_embed.detach(), (4, 4), (5, 6))  # 计算并保存当前步骤的中间状态。
assert resized_position24.shape == (1, 31, 24)  # 用受控断言验证关键不变量。
assert torch.equal(resized_position24[:, :1], model24.pos_embed.detach()[:, :1])  # 用受控断言验证关键不变量。
assert torch.isfinite(resized_position24).all()  # 用受控断言验证关键不变量。

# 非恒定坐标坡度 fixture：插值后仍应保持从上到下、从左到右的空间顺序。
fixture_grid24 = (torch.arange(2, dtype=torch.float32)[:, None] * 10  # 计算并保存当前步骤的中间状态。
                  + torch.arange(3, dtype=torch.float32)[None, :])  # 计算并保存当前步骤的中间状态。
fixture_position24 = torch.cat([torch.tensor([[[-7.0]]]),  # 计算并保存当前步骤的中间状态。
                                fixture_grid24.reshape(1, 6, 1)], dim=1)  # 计算并保存当前步骤的中间状态。
fixture_resized24 = resize_position_embedding(fixture_position24, (2, 3), (4, 6))  # 计算并保存当前步骤的中间状态。
fixture_map24 = fixture_resized24[:, 1:].reshape(4, 6)  # 计算并保存当前步骤的中间状态。
assert fixture_resized24.shape == (1, 25, 1)  # 用受控断言验证关键不变量。
assert fixture_resized24[0, 0, 0].item() == -7.0  # 用受控断言验证关键不变量。
assert torch.all(fixture_map24[1:].mean(1) > fixture_map24[:-1].mean(1))  # 用受控断言验证关键不变量。
assert torch.all(fixture_map24[:, 1:].mean(0) > fixture_map24[:, :-1].mean(0))  # 用受控断言验证关键不变量。
assert fixture_map24[0, 0] < fixture_map24[-1, -1]  # 用受控断言验证关键不变量。

# 常量场是反例合同：正确插值应保持常量，不能要求任意输入都必须发生数值改变。
constant_position24 = torch.cat([torch.tensor([[[3.0]]]), torch.ones(1, 6, 1)], dim=1)  # 计算并保存当前步骤的中间状态。
constant_resized24 = resize_position_embedding(constant_position24, (2, 3), (4, 6))  # 计算并保存当前步骤的中间状态。
assert constant_resized24[0, 0, 0].item() == 3.0  # 用受控断言验证关键不变量。
assert torch.allclose(constant_resized24[:, 1:], torch.ones(1, 24, 1), atol=1e-6)  # 用受控断言验证关键不变量。


## 13. 梯度健康、state_dict 指纹与发布合同

一次 probe backward 检查 patch projection、QKV、位置向量和 head 都获得有限非零梯度。位置向量没有梯度往往说明 token 拼接/参数注册出错；QKV 无梯度则可能是错误 detach 或未走 attention。

manifest 绑定架构超参数、输入、类别、优化器、训练 seed 和权重摘要。SHA-256 用来发现制品是否改变，不代替可信签名和安全加载；不应加载来源不明的 pickle checkpoint。


In [ ]:
model24.train()  # 执行当前语句以推进本节示例。
optimizer24.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
probe_loss24 = criterion24(model24(train_x24[:6]), train_y24[:6])  # 计算并保存当前步骤的中间状态。
probe_loss24.backward()  # 执行当前语句以推进本节示例。
gradients24 = {name: float(parameter.grad.norm()) for name, parameter in model24.named_parameters()  # 计算并保存当前步骤的中间状态。
               if parameter.grad is not None}  # 按当前条件选择后续控制路径。

def state_fingerprint24(model):  # 定义本节可复用的核心函数。
    digest = sha256()  # 计算并保存当前步骤的中间状态。
    for name, tensor in sorted(model.state_dict().items()):  # 遍历输入元素以累积或检查结果。
        value = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        digest.update(name.encode("utf-8"))  # 执行当前语句以推进本节示例。
        digest.update(str(value.dtype).encode("ascii"))  # 执行当前语句以推进本节示例。
        digest.update(str(tuple(value.shape)).encode("ascii"))  # 执行当前语句以推进本节示例。
        digest.update(value.numpy().tobytes())  # 执行当前语句以推进本节示例。
    return digest.hexdigest()  # 返回当前分支计算出的结果。

fingerprint24 = state_fingerprint24(model24)  # 计算并保存当前步骤的中间状态。
manifest24 = {  # 计算并保存当前步骤的中间状态。
    "artifact": "vit-p4-d24-h4-l2-toy-v1",  # 执行当前语句以推进本节示例。
    "architecture": {"image_size": [16, 16], "patch_size": [4, 4],  # 执行当前语句以推进本节示例。
                     "embed_dim": 24, "heads": 4, "depth": 2, "mlp_ratio": 2.0},  # 执行当前语句以推进本节示例。
    "input": {"layout": "NCHW", "channels": 1, "range": [0.0, 1.0]},  # 执行当前语句以推进本节示例。
    "classes": ["top_left", "top_right", "bottom_center"],  # 执行当前语句以推进本节示例。
    "optimizer": {"name": "AdamW", "lr": 0.01, "weight_decay": 0.01,  # 执行当前语句以推进本节示例。
                  "overfit_steps": 120},  # 执行当前语句以推进本节示例。
    "seed": SEED, "torch": torch.__version__, "state_dict_sha256": fingerprint24,  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。

assert gradients24["patch_embed.projection.weight"] > 0  # 用受控断言验证关键不变量。
assert gradients24["cls_token"] > 0  # 用受控断言验证关键不变量。
assert gradients24["pos_embed"] > 0  # 用受控断言验证关键不变量。
assert gradients24["blocks.0.attention.qkv.weight"] > 0  # 用受控断言验证关键不变量。
assert gradients24["head.weight"] > 0  # 用受控断言验证关键不变量。
assert all(math.isfinite(value) for value in gradients24.values())  # 用受控断言验证关键不变量。
assert len(fingerprint24) == 64  # 用受控断言验证关键不变量。
assert state_fingerprint24(model24) == fingerprint24  # 用受控断言验证关键不变量。
print(json.dumps(manifest24, ensure_ascii=False, indent=2))  # 计算并保存当前步骤的中间状态。


## 14. 生产边界与资料

真实 ViT 工程还需要：来源/时间切分防泄漏；训练集统计量和增强版本；随机裁剪、Mixup/CutMix 等标签合同；warmup、scheduler、参数分组与梯度缩放；大分辨率 attention 的 $O(T^2)$ 显存；混合精度稳定性；动态 batch；位置插值回归；量化/导出算子一致性；校准、OOD、漂移和类别分群；模型卡与回滚。

若需要可变分辨率，可考虑固定二维位置函数、相对位置偏置或旋转位置编码等设计，但这会改变架构，必须重新训练/验证。注意力可视化也不能替代真实的错误分析和反事实测试。

原始论文与官方资料：

- Vaswani 等，[Attention Is All You Need](https://arxiv.org/abs/1706.03762)，scaled dot-product 与多头注意力来源。
- Dosovitskiy 等，[An Image is Worth 16x16 Words](https://arxiv.org/abs/2010.11929)，Vision Transformer。
- Touvron 等，[Training data-efficient image transformers & distillation through attention](https://arxiv.org/abs/2012.12877)，DeiT 训练视角。
- PyTorch 官方文档：[LayerNorm](https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)、[softmax](https://pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html)、[`interpolate`](https://pytorch.org/docs/stable/generated/torch.nn.functional.interpolate.html)。

结论边界：本 notebook 证明手写 ViT 核心层在受控 16×16 位置任务上满足 shape、softmax、梯度与小样本过拟合测试；不证明 ImageNet 复现、真实场景泛化、注意力解释忠实度或生产性能。
